In [3]:
import pandas as pd
import numpy as np
import time

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

housing = fetch_california_housing()

df = pd.DataFrame(housing.data, columns=housing.feature_names)

target = pd.Series(housing.target, name="MedHouseVal")

df = pd.concat([df, target], axis=1)

df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [4]:
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (16512, 8)
Testing data: (4128, 8)


In [5]:
correlation = df.corr()["MedHouseVal"].drop("MedHouseVal")

correlation = correlation.abs().sort_values(ascending=False)

print("Features ranked by correlation:")
print(correlation)

Features ranked by correlation:
MedInc        0.688075
AveRooms      0.151948
Latitude      0.144160
HouseAge      0.105623
AveBedrms     0.046701
Longitude     0.045967
Population    0.024650
AveOccup      0.023737
Name: MedHouseVal, dtype: float64


In [6]:
# Model A: top 3 features
features_A = correlation.head(3).index.tolist()

# Model B: top 5 features
features_B = correlation.head(5).index.tolist()

# Model C: all features
features_C = X.columns.tolist()

print("Model A:", features_A)
print("Model B:", features_B)
print("Model C:", features_C)

Model A: ['MedInc', 'AveRooms', 'Latitude']
Model B: ['MedInc', 'AveRooms', 'Latitude', 'HouseAge', 'AveBedrms']
Model C: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


In [7]:
model_A = LinearRegression()
model_B = LinearRegression()
model_C = LinearRegression()

start = time.time()
model_A.fit(X_train[features_A], y_train)
time_A = time.time() - start

start = time.time()
model_B.fit(X_train[features_B], y_train)
time_B = time.time() - start

start = time.time()
model_C.fit(X_train[features_C], y_train)
time_C = time.time() - start

print("Model A training time:", time_A)
print("Model B training time:", time_B)
print("Model C training time:", time_C)

Model A training time: 0.07506322860717773
Model B training time: 0.0013935565948486328
Model C training time: 0.018158912658691406


In [9]:
pred_A = model_A.predict(X_test[features_A])
pred_B = model_B.predict(X_test[features_B])
pred_C = model_C.predict(X_test[features_C])

mae_A = mean_absolute_error(y_test, pred_A)
mse_A = mean_squared_error(y_test, pred_A)
rmse_A = np.sqrt(mse_A)
r2_A = r2_score(y_test, pred_A)

mae_B = mean_absolute_error(y_test, pred_B)
mse_B = mean_squared_error(y_test, pred_B)
rmse_B = np.sqrt(mse_B)
r2_B = r2_score(y_test, pred_B)

mae_C = mean_absolute_error(y_test, pred_C)
mse_C = mean_squared_error(y_test, pred_C)
rmse_C = np.sqrt(mse_C)
r2_C = r2_score(y_test, pred_C)

In [10]:
results = pd.DataFrame({
    "Model": ["Model A", "Model B", "Model C"],
    "Features": [3, 5, 8],
    "MAE": [mae_A, mae_B, mae_C],
    "MSE": [mse_A, mse_B, mse_C],
    "RMSE": [rmse_A, rmse_B, rmse_C],
    "R2 Score": [r2_A, r2_B, r2_C],
    "Training Time": [time_A, time_B, time_C]
})

results.round(4)

,Model,Features,MAE,MSE,RMSE,R2 Score,Training Time
0,Model A,3,0.6215,0.6942,0.8332,0.4703,0.0751
1,Model B,5,0.5797,0.6383,0.7989,0.5129,0.0014
2,Model C,8,0.5332,0.5559,0.7456,0.5758,0.0182


In [11]:
best_model = results.loc[results["R2 Score"].idxmax(), "Model"]

print("Best performing model:", best_model)

print("\nFeature combinations:")
print("Model A:", features_A)
print("Model B:", features_B)
print("Model C:", features_C)

print("\nTraining times:")
print("Model A:", time_A, "seconds")
print("Model B:", time_B, "seconds")
print("Model C:", time_C, "seconds")

Best performing model: Model C

Feature combinations:
Model A: ['MedInc', 'AveRooms', 'Latitude']
Model B: ['MedInc', 'AveRooms', 'Latitude', 'HouseAge', 'AveBedrms']
Model C: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Training times:
Model A: 0.07506322860717773 seconds
Model B: 0.0013935565948486328 seconds
Model C: 0.018158912658691406 seconds
